In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Load CSV dataset
df = pd.read_csv('dataset/dataset_csv_split/train_cleaned_medical_claims.csv')

df = df.dropna(subset=['text'])

# Extract raw text and labels
X = df['text'].astype(str)
y = df['label']

# Recreate the 70/15/15 Split
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

# TF-IDF Vectorization
# Ignore standard English stopwords
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# 'Fit' learns the vocabulary from the training data, 'transform' turns it into vectors
X_train_tfidf = vectorizer.fit_transform(X_train)

# Only 'transform' the validation and test sets to prevent data leakage
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

# Train and initialize baseline model (Logistic Regression Model)
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

# Quantitative results: test on unseen data (test set)
y_pred = lr_model.predict(X_test_tfidf)

print("Quantitative Results: ")
print(classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))
print(f"\nOverall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

# Quantitative results: see which words are most strongly associated with each class (Fake vs Real)
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = lr_model.coef_[0]
sorted_indices = np.argsort(coefficients)
top_fake_words = feature_names[sorted_indices[-10:]]
top_real_words = feature_names[sorted_indices[:10]]

print("Qualitative Results: ")
print("\nWords most strongly flagged as FAKE misinformation:")
for word in reversed(top_fake_words):
    print(f"- {word}")

print("\nWords most strongly flagged as REAL medical facts:")
for word in top_real_words:
    print(f"- {word}")

Quantitative Results: 
              precision    recall  f1-score   support

    Real (0)       0.96      0.99      0.97        79
    Fake (1)       0.99      0.97      0.98        92

    accuracy                           0.98       171
   macro avg       0.98      0.98      0.98       171
weighted avg       0.98      0.98      0.98       171


Overall Accuracy: 97.66%

Qualitative Results: 

Words most strongly flagged as FAKE misinformation:
- water
- cure
- 19
- baby
- control
- cinnamon
- covid
- checkups
- depression
- cures

Words most strongly flagged as REAL medical facts:
- covid19
- effects
- ly
- bit
- cause
- help
- https
- outcomes
- health
- helps
